# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Moezulhaq24/FlyRank-Internship-ML/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

My chosen lane is Refresh / Content Opportunity Scoring.

This is mainly a scoring and ranking task. The goal is not only to say whether a page is good or bad. The goal is to give each content page a refresh opportunity score and then rank the pages in priority order.

This ranking can help SEO analysts or content editors decide which pages should be reviewed first for refresh, CTR improvement, content expansion, engagement improvement, or monitoring.

One row represents one content page, and the model should use signals like CTR, engagement rate, impressions, average position, content age, and trend direction to estimate review priority.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Target or proxy

The ideal target is refresh opportunity: which pages are most worth reviewing or refreshing first.

This exact target is not directly available in the dataset, so I will create a proxy label. A proxy label is a measurable signal that is close to the real decision.

For this lane, a page can be treated as a refresh opportunity if it shows signals such as:

- declining trend with search demand
- low CTR but enough impressions
- old content with visibility
- weak engagement on pages that still receive sessions

This proxy is not perfect, but it gives a practical starting point for scoring and ranking pages.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np

DATA_PATH = "content_refresh_anonymized.csv"

df = pd.read_csv(DATA_PATH)

df["trend_clean"] = df["trend_direction"].astype(str).str.lower().str.strip()

declining_with_demand = (
    (df["trend_clean"] == "down") &
    (df["impressions_90d"] >= 100)
)

low_ctr_visible = (
    (df["impressions_90d"] >= 500) &
    (df["avg_position"] > 0) &
    (df["avg_position"] <= 20) &
    (df["ctr"] < 0.5)
)

old_visible_page = (
    (df["content_age_days"] >= 365) &
    (df["impressions_90d"] >= 100)
)

weak_engagement_visible = (
    (df["sessions_90d"] >= 30) &
    (df["engagement_rate"] < 30)
)

df["refresh_opportunity_proxy"] = (
    declining_with_demand |
    low_ctr_visible |
    old_visible_page |
    weak_engagement_visible
).astype(int)

df["refresh_opportunity_proxy"].value_counts()

,count
refresh_opportunity_proxy,
1,20067
0,9933


## 3. Success metric

The success metric I would use is Precision@K, such as Precision@20 or Precision@50.

This is suitable because the real action is a ranked review queue. Content teams usually cannot review every page, so the important question is:

Out of the top K pages recommended by the model, how many are actually useful refresh candidates?

For this lane, Precision@50 is a good metric because it measures the quality of the top 50 pages that an SEO analyst or content editor may review first.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

proxy_rate = df["refresh_opportunity_proxy"].mean()

print("Total pages:", len(df))
print("Refresh opportunity proxy pages:", df["refresh_opportunity_proxy"].sum())
print("Proxy positive rate:", round(proxy_rate, 3))


Total pages: 30000
Refresh opportunity proxy pages: 20067
Proxy positive rate: 0.669


## 4. The unit of analysis, as a real dataframe

In the dataframe, one row represents one page with its search, engagement, ranking, and freshness signals. These signals can be used to score and rank pages for refresh review.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
lane_columns = [
    "client_id",
    "impressions_90d",
    "ctr",
    "avg_position",
    "sessions_90d",
    "engagement_rate",
    "content_age_days",
    "trend_direction",
    "refresh_opportunity_proxy"
]

lane_df = df[lane_columns].copy()

print("Shape of lane dataframe:", lane_df.shape)
lane_df.head(10)

Shape of lane dataframe: (30000, 9)


,client_id,impressions_90d,ctr,avg_position,sessions_90d,engagement_rate,content_age_days,trend_direction,refresh_opportunity_proxy
0,client_f369cb89fc,3803,0.76,10.6,17,5.88,187,down,1
1,client_4e07408562,15320,0.05,20.3,9,0.00,445,down,1
2,client_7f2253d7e2,12581,0.09,36.5,11,0.00,141,down,1
3,client_19581e27de,11751,0.49,6.2,78,1.28,463,stable,1
4,client_3fdba35f04,19140,0.13,44.0,145,0.00,263,down,1
5,client_f369cb89fc,3970,0.03,8.5,5,0.00,147,down,1
6,client_8722616204,20,0.00,7.0,1,0.00,90,down,0
7,client_19581e27de,1724,0.06,21.2,28,3.57,445,stable,1
8,client_6208ef0f77,32574,0.09,46.0,68,5.88,90,down,1
9,client_19581e27de,1240,0.16,4.9,3,0.00,257,down,1


## 5. Why ML beats a fixed rule here

A fixed rule is too simple for this problem.

For example, a rule like "if a page is old, refresh it" does not make enough sense by itself. A page can be old but still performing well. Another page can be newer but declining, visible in search, and getting low CTR.

Refresh opportunity depends on multiple signals together, such as impressions, CTR, average position, trend direction, engagement rate, sessions, and content age.

In Assignment 1, the hand-written rule performed much weaker than the learned model. The simple rule gave around 24% useful results, while the learned model reached around 74% in the evaluated ranking. This shows that a model can learn better patterns from multiple features than one fixed rule.

So ML is useful here because the goal is decision-support: helping humans review the most promising pages first, not making automatic content decisions.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.